# Chapitre 7 - Exercices 7.1 à 7.5

Ces exercices établissent les résultats théoriques sur lesquels reposent les critères
analytiques et la validation croisée du chapitre :

* **7.1** : l'estimation de l'erreur intra-échantillon (7.24) ;
* **7.2** : la décomposition de l'erreur de classement et l'interaction biais-variance en
  perte 0-1 ;
* **7.3** : la formule leave-one-out sans réajustement pour un lisseur linéaire ;
* **7.4** : l'optimisme moyen $\frac2N\sum\mathrm{Cov}(\hat y_i, y_i)$ ;
* **7.5** : $\sum\mathrm{Cov}(\hat y_i, y_i) = \mathrm{trace}(\mathbf S)\sigma^2_\varepsilon$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch07_selection as sel
set_style()

## Exercice 7.1 : Estimation de l'erreur intra-échantillon

**Énoncé.** Établir l'estimation (7.24) de l'erreur intra-échantillon :
$\mathbb E_{\mathbf y}(\mathrm{Err}_{\mathrm{in}}) = \mathbb E_{\mathbf y}(\overline{\mathrm{err}}) + 2\frac dN\sigma^2_\varepsilon$
pour un ajustement linéaire à $d$ paramètres (fonctions de base fixées) dans le modèle
$Y = f(X) + \varepsilon$.

**Cadre et notations.** Les entrées $x_1, \dots, x_N$ sont fixées. On note
$\mathbf f = (f(x_1), \dots, f(x_N))^T$, $\mathbf y = \mathbf f + \boldsymbol\varepsilon$ avec
$\boldsymbol\varepsilon$ centré, de matrice de covariance $\sigma^2_\varepsilon I_N$. Le
modèle est ajusté par moindres carrés sur $d$ fonctions de base linéairement
indépendantes : $\hat{\mathbf y} = \mathbf H\mathbf y$ où $\mathbf H$ est la matrice
chapeau, **projecteur orthogonal** de rang $d$ ($\mathbf H^T = \mathbf H$, $\mathbf H^2 = \mathbf H$,
$\mathrm{trace}(\mathbf H) = d$).

Pour la perte quadratique,
$\overline{\mathrm{err}} = \frac1N\|\mathbf y - \hat{\mathbf y}\|^2$ et
$\mathrm{Err}_{\mathrm{in}} = \frac1N\mathbb E_{\mathbf Y^0}\|\mathbf Y^0 - \hat{\mathbf y}\|^2$,
où $\mathbf Y^0 = \mathbf f + \boldsymbol\varepsilon^0$ est un nouveau vecteur de réponses,
indépendant de $\mathbf y$.

**Lemme (espérance d'une forme quadratique).** Pour toute matrice $\mathbf A$ (non
aléatoire), $\mathbb E\|\mathbf A\mathbf y\|^2 = \|\mathbf A\mathbf f\|^2 + \sigma^2_\varepsilon\,\mathrm{trace}(\mathbf A^T\mathbf A)$.

*Preuve.* On développe $\|\mathbf A\mathbf f + \mathbf A\boldsymbol\varepsilon\|^2 = \|\mathbf A\mathbf f\|^2 + 2\mathbf f^T\mathbf A^T\mathbf A\boldsymbol\varepsilon + \boldsymbol\varepsilon^T\mathbf A^T\mathbf A\boldsymbol\varepsilon$.
Le terme du milieu est d'espérance nulle (linéarité, $\mathbb E\boldsymbol\varepsilon = 0$).
Pour le dernier, un scalaire est égal à sa trace et la trace est invariante par
permutation circulaire :
$\mathbb E[\boldsymbol\varepsilon^T\mathbf M\boldsymbol\varepsilon] = \mathbb E[\mathrm{trace}(\mathbf M\boldsymbol\varepsilon\boldsymbol\varepsilon^T)] = \mathrm{trace}(\mathbf M\,\mathbb E[\boldsymbol\varepsilon\boldsymbol\varepsilon^T]) = \sigma^2_\varepsilon\mathrm{trace}(\mathbf M)$
avec $\mathbf M = \mathbf A^T\mathbf A$. $\square$

**Erreur d'apprentissage.** $\mathbf y - \hat{\mathbf y} = (I - \mathbf H)\mathbf y$, et
$I - \mathbf H$ est aussi un projecteur orthogonal, de rang $N - d$, donc
$(I - \mathbf H)^T(I - \mathbf H) = I - \mathbf H$ et $\mathrm{trace}(I - \mathbf H) = N - d$.
Le lemme donne

$$\mathbb E_{\mathbf y}(\overline{\mathrm{err}}) = \frac1N\|(I - \mathbf H)\mathbf f\|^2 + \frac{N - d}N\sigma^2_\varepsilon.$$

**Erreur intra-échantillon.** On écrit $\mathbf Y^0 - \hat{\mathbf y} = \boldsymbol\varepsilon^0 + (\mathbf f - \mathbf H\mathbf y)$.
Comme $\boldsymbol\varepsilon^0$ est centré et indépendant de $\mathbf y$, le double produit
est d'espérance nulle et
$\mathbb E_{\mathbf Y^0}\|\mathbf Y^0 - \hat{\mathbf y}\|^2 = N\sigma^2_\varepsilon + \|\mathbf f - \mathbf H\mathbf y\|^2$.
Puis, en écrivant $\mathbf f - \mathbf H\mathbf y = (I - \mathbf H)\mathbf f - \mathbf H\boldsymbol\varepsilon$
et en appliquant le même raisonnement ($\mathbf H^T\mathbf H = \mathbf H$, de trace $d$) :

$$\mathbb E_{\mathbf y}(\mathrm{Err}_{\mathrm{in}}) = \sigma^2_\varepsilon + \frac1N\|(I - \mathbf H)\mathbf f\|^2 + \frac dN\sigma^2_\varepsilon.$$

**Conclusion.** Par différence, le terme de biais $\frac1N\|(I - \mathbf H)\mathbf f\|^2$
disparaît :

$$\mathbb E_{\mathbf y}(\mathrm{Err}_{\mathrm{in}}) - \mathbb E_{\mathbf y}(\overline{\mathrm{err}})
= \sigma^2_\varepsilon + \frac dN\sigma^2_\varepsilon - \frac{N - d}N\sigma^2_\varepsilon = 2\cdot\frac dN\,\sigma^2_\varepsilon,$$

ce qui est exactement (7.24). L'erreur d'apprentissage sous-estime le bruit d'un facteur
$(N - d)/N$, et l'erreur intra-échantillon le surestime de $d/N$ (variance de
l'estimateur) : l'écart total est $2d\sigma^2_\varepsilon/N$.

**Vérification numérique.** Régression polynomiale de degré $d - 1$ sur $N = 30$ points
fixés, $f(x) = e^{x}\cos(3x)$, $\sigma_\varepsilon = 0{,}5$. On compare les espérances
simulées des deux erreurs aux formules exactes ci-dessus.